In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.6 Speculative decoding

> 🎯 **Goal:** Let a cheap draft model propose several tokens that the expensive target model verifies in one pass, accepting the longest correct run, to cut the number of expensive steps.

**Why this matters.** Generation is sequential and the per-token cost of a big model is the dominant latency in serving. Speculative decoding attacks that cost without changing the output: it lets you produce several tokens for roughly the price of one big-model step, whenever a small model can guess what the big one was going to say anyway. It is one of the highest-leverage latency wins in modern inference.

**The intuition.** Picture a senior engineer and a fast junior. The junior (the **draft** model) is quick but not always right, so they sketch the next few words. The senior (the **target** model) is slow but authoritative, and can check the whole sketch in a single glance. Wherever the junior matched what the senior would have written, the senior just accepts it, no need to write it out themselves. The moment the junior is wrong, the senior corrects that one token and the next round begins. You get the senior's quality at the junior's speed, for the part they agree on.

**The mechanics.** This is a concept demo at pocket scale: both models are tiny `train_tiny` models (a stronger `target` and a deliberately different, weaker `draft`), so the point is the *protocol*, not a real speedup. The draft greedily proposes 4 tokens. Then the target, in verification, checks each proposed token: if the target's own greedy next-token at that position matches the draft's guess, accept it and move on; at the first mismatch, stop. The accepted prefix is exactly what the target would have produced on its own, so the output distribution is unchanged. Real systems use a probabilistic accept/reject rule to preserve the distribution even with sampling, but the greedy version here shows the core idea: every accepted token skipped a separate expensive target step.

In [ ]:
from pocketlm import train_tiny, pick_config

corpus = open(CORPUS_PATH, encoding="utf-8").read()
target, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
draft, _ = train_tiny(corpus, pick_config(DEVICE, 1), seed=1)   # a different, weaker model
ctx = torch.tensor([tok.encode("The ")])
proposed = []
d = ctx.clone()
for _ in range(4):                       # draft proposes 4 tokens greedily
    dl, _ = draft(d[:, -draft.cfg.block_size:])
    nt = dl[:, -1].argmax(-1, keepdim=True)
    d = torch.cat([d, nt], dim=1)
    proposed.append(int(nt.item()))
accepted = 0
t = ctx.clone()
for tok_id in proposed:                  # target verifies greedily
    tl, _ = target(t[:, -target.cfg.block_size:])
    if int(tl[:, -1].argmax(-1).item()) == tok_id:
        accepted += 1
        t = torch.cat([t, torch.tensor([[tok_id]])], dim=1)
    else:
        break
print(f"draft proposed {len(proposed)}, target accepted {accepted}")

**What just happened.** The draft proposed 4 tokens and the target accepted some prefix of them. Each accepted token is one expensive target step you did not have to run sequentially, because the target verified them together. The target always has the final say (it stops at the first disagreement and would correct it), so correctness is preserved exactly. The speedup is real only when agreement is high, and at pocket scale with two tiny models you should treat the accept count as an illustration of the protocol, not a benchmark.

⚠️ **Common pitfalls**
- Believing the output changes. It does not: the accepted tokens are exactly what the target would have generated alone, that guarantee is the whole appeal.
- Using a draft that rarely agrees with the target. Then almost nothing is accepted and you pay for the draft for no gain, so the draft must be cheap *and* well-aligned.
- Proposing too many tokens per round. A long wrong guess wastes draft work, there is an optimal draft length that depends on the agreement rate.

🏋️ **Try it yourself.** Change the draft's seed (currently 1) or train it on the same seed as the target and watch the accepted count rise as the two models align. Then increase the proposal length from 4 to 8 and see whether the extra proposals get accepted or thrown away.

In [ ]:
assert 0 <= accepted <= len(proposed)